# **Problemas de pilas y colas usando MM1**

# **HEcho por rene Zapata Hernandez.**

el problema que se simila con el codigo es el siguiente:

El Banco de Colombia está interesado en medir el desempeño de sus cajeros durante las operaciones de pagos y retiros, pues no cuenta con cajeros electrónicos y considera conveniente tener cajas disponibles para que los ciudadanos realicen sus operaciones sin demora. El banco desea saber si debe asignar una caja exclusiva para retiros y dos para pagos, o viceversa, esto con el propósito de adoptar las adecuaciones necesarias en miras de prestar el mejor servicio posible a sus usuarios.

Para resolver este caso, presta atención a la siguiente información:

El banco dispone de tres cajas para ambas finalidades (pagos y retiros), las cuales prestan su servicio de acuerdo con el tipo de usuario.  

En la Tabla 1 se definen los tipos de usuario, detallando los tiempos de llegada y servicio estimados para cada uno con base en la acción que deseen realizar. Ten en cuenta que el 70 % de los usuarios hace retiros, mientras que los demás realizan consignaciones o pagos.

Por otro lado, los cajeros se deben simular como modelos M/M/1, pues son independientes. Además, se asume que la velocidad de atención es igual para todos los cajeros (inmediata) y la velocidad de desplazamiento es despreciable.  

El banco opera 8 horas al día. Se puede asumir que los días son iguales y establecer las sugerencias necesarias a partir de la modelación de un solo día y sus réplicas.

Ejecuta al menos 10 corridas del modelo y calcula las estadísticas necesarias para resolver los siguientes puntos:

Calcula las estadísticas necesarias para identificar el cajero con menor y mayor tiempo promedio de atención (no es necesario segregar los usuarios).

Establece el promedio de usuarios de cada tipo en la totalidad de cajeros.

Determina el total de usuarios de cada tipo en cada una de las réplicas y detalla el modelo con menor cantidad de usuarios por tipo.

Define si es necesario crear un nuevo cajero utilizando los tiempos promedio de espera en todos los criterios del modelo.

Decide cuántos cajeros deben ofrecer atención exclusiva para pagos y cuántos para retiros.

Los detalles de atención de cada cajero y servicio se definen a continuación:
Tipo de acción	Tipo de usuario	Exponencial de uso del servicio	Exponencial de media de llegada
Retiro	Rápido	1 minutos	1 minutos
	Normal	2 minutos	2 minutos
	Lento	3 minutos	3 minutos
	Muy lento	4 minutos	3 minutos
Consignación o pago	Rápido	3 minutos	1 minutos
	Normal	3 minutos	2 minutos
	Lento	5 minutos	3 minutos
	Muy lento	7 minutos	4 minutos
Las probabilidades de los tipos de usuario se detallan en la siguiente tabla:

Tipo de acción	Tipo de usuario	Probabilidad
Retiro	Rápido	0,23
	Normal	0,40
	Lento	0,17
	Muy lento	0,20
Consignación o pago	Rápido	0,10
	Normal	0,20
	Lento	0,30
	Muy lento	0,40



In [2]:
!pip install simpy

In [13]:
#empieza la simulacion de pilas

import random
import simpy
import numpy as np
import pandas as pd

SEMINAS_REPLICAS = 500
TIEMPO_SIMULACION = 480  # 8 horas en minutos

# Estructura de datos para los tiempos (Media de servicio, Media de llegada)
DATOS_CLIENTES = {
    'Retiro': {
        'Rápido':     {'mu': 1, 'lambda_inv': 1, 'prob': 0.23},
        'Normal':     {'mu': 2, 'lambda_inv': 2, 'prob': 0.40},
        'Lento':      {'mu': 3, 'lambda_inv': 3, 'prob': 0.17},
        'Muy lento':  {'mu': 4, 'lambda_inv': 3, 'prob': 0.20}
    },
    'Pago': {
        'Rápido':     {'mu': 3, 'lambda_inv': 1, 'prob': 0.10},
        'Normal':     {'mu': 3, 'lambda_inv': 2, 'prob': 0.20},
        'Lento':      {'mu': 5, 'lambda_inv': 3, 'prob': 0.30},
        'Muy lento':  {'mu': 7, 'lambda_inv': 4, 'prob': 0.40}
    }
}

# --- GENERADOR DE PROCESOS SIMPY ---
class BancoSimulacion:
    def __init__(self, env, num_cajeros_retiro, num_cajeros_pago):
        self.env = env
        # Definimos los recursos (Cajeros M/M/1 independientes compartiendo fila por tipo de acción)
        self.cajeros_retiro = simpy.Resource(env, capacity=num_cajeros_retiro)
        self.cajeros_pago = simpy.Resource(env, capacity=num_cajeros_pago)

        # Almacenamiento de métricas por réplica
        self.records = []
    def cliente(self, tipo_accion, sub_tipo):
        llega = self.env.now
        stats = {
            'tipo_accion': tipo_accion,
            'sub_tipo': sub_tipo,
            'tiempo_llegada': llega,
            'tiempo_espera': 0,
            'tiempo_servicio': 0,
            'cajero_id': None
        }

        # Determinar a qué cola va
        recurso = self.cajeros_retiro if tipo_accion == 'Retiro' else self.cajeros_pago

        # Solicitar cajero
        with recurso.request() as peticion:
            yield peticion
            espera = self.env.now - llega
            stats['tiempo_espera'] = espera

            # Tiempo de servicio exponencial
            media_servicio = DATOS_CLIENTES[tipo_accion][sub_tipo]['mu']
            servicio = random.expovariate(1.0 / media_servicio)
            stats['tiempo_servicio'] = servicio

            yield self.env.timeout(servicio)

        self.records.append(stats)
#fin de la clase


def generador_llegadas(env, banco):
          while True:
            # 1. Determinar tipo de acción (70% Retiro, 30% Pago)
            tipo_accion = 'Retiro' if random.random() < 0.70 else 'Pago'
            # 2. Determinar sub-tipo según sus probabilidades
            sub_tipos = list(DATOS_CLIENTES[tipo_accion].keys())
            probabilidades = [DATOS_CLIENTES[tipo_accion][st]['prob'] for st in sub_tipos]
            sub_tipo = random.choices(sub_tipos, weights=probabilidades)[0]
            # 3. Tiempo de llegada exponencial
            media_llegada = DATOS_CLIENTES[tipo_accion][sub_tipo]['lambda_inv']
            tiempo_entre_llegada = random.expovariate(1.0 / media_llegada)
            yield env.timeout(tiempo_entre_llegada)
            # Activar el proceso del cliente en el entorno
            env.process(banco.cliente(tipo_accion, sub_tipo))

# --- EJECUCIÓN DE RÉPLICAS ---
def ejecutar_simulacion(cajeros_retiro, cajeros_pago):
          resultados_replicas = []
          for replica in range(SEMINAS_REPLICAS):
            random.seed(replica) # Para reproducibilidad
            env = simpy.Environment()
            banco = BancoSimulacion(env, num_cajeros_retiro=cajeros_retiro, num_cajeros_pago=cajeros_pago)
            env.process(generador_llegadas(env, banco))
            env.run(until=TIEMPO_SIMULACION)
            # Convertir récords de la réplica actual a DataFrame
            df_rep = pd.DataFrame(banco.records)
            df_rep['replica'] = replica
            resultados_replicas.append(df_rep)

          return pd.concat(resultados_replicas, ignore_index=True)

# Correr ambos escenarios para comparar
#BS= new BancoSimulacion()
print("Simulando Escenario A (2 Retiros, 1 Pago)...")
df_escenario_A = ejecutar_simulacion(cajeros_retiro=2, cajeros_pago=1)
print(df_escenario_A)
print("Simulando Escenario B (1 Retiro, 2 Pagos)...")
df_escenario_B = ejecutar_simulacion(cajeros_retiro=1, cajeros_pago=2)
print(df_escenario_B)

#identifico los tiempos de atencion (Mayor y Menor)
print("\nIdentifico tiempo de atencion")
print(df_escenario_A.groupby('tipo_accion')['tiempo_servicio'].mean())


#Promedio de usuarios por tipo en la totalidad de cajeros
print("\n Promedio de usuarios por tipo en la totalidad de cajeros")
usuarios_por_dia = df_escenario_A.groupby(['replica', 'tipo_accion', 'sub_tipo']).size().unstack(fill_value=0)
print(usuarios_por_dia.mean())

#El modelo con menor cantidad por tipo (estabilidad)
conteo_por_replica = df_escenario_A.groupby(['replica', 'tipo_accion']).size().unstack()
print(f"Réplica con menos retiros: {conteo_por_replica['Retiro'].idxmin()}")
print(f"Réplica con menos pagos: {conteo_por_replica['Pago'].idxmin()}")

#Es necesario crear u nievo cajero ?(criterio de saturacion): se calcula el tiempo promedio deespera, si la tasa de llegada (lamda) se aserca o supera la asa de servicio (mu) el tiempo de espera iende a inficnito
print("Espera Promedio Escenario A:\n", df_escenario_A.groupby('tipo_accion')['tiempo_espera'].mean())
print("\nEspera Promedio Escenario B:\n", df_escenario_B.groupby('tipo_accion')['tiempo_espera'].mean())


Simulando Escenario A (2 Retiros, 1 Pago)...
      tipo_accion   sub_tipo  tiempo_llegada  tiempo_espera  tiempo_servicio  \
0          Retiro     Normal        4.972190       0.000000         1.926602   
1          Retiro     Normal        3.221019       0.000000         4.774389   
2          Retiro     Normal        7.791760       0.000000         3.323750   
3            Pago  Muy lento        2.182853       0.000000        10.720814   
4          Retiro     Normal       24.116177       0.000000         1.296385   
...           ...        ...             ...            ...              ...   
98692      Retiro     Rápido      469.269446       1.125664         1.235251   
98693      Retiro     Normal      468.316039       0.000000         5.529831   
98694      Retiro     Normal      475.811188       0.000000         0.917945   
98695      Retiro      Lento      475.275833       0.000000         2.376638   
98696      Retiro     Normal      477.075756       0.576716         1.02238

Fenomeno detectado:
Con u solo cajero para pagos (escenarioa) comoel 40% de los usarios de pago son muy lentos (demoran 7 minutos) la tasa acumulada global acumula mucha gente, el tiempo de espera se vuelve insostenible (la cola no se vacia al final de las 8 horas).
y si se deja un solo cajero para retiros: aunque se representan el 70% de los clientes, sustiempos de atencion son muy veloces (1 a 4 minutos) solo cajero podria hacer el trabajo de los que 1 solo cajero haciendo pagos.

El cuello de botella se presenta en laoperacion de pagos